In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, auc
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/'
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2','IPS']

In [ ]:
class_df = pd.read_csv(f'{projdir}/csv/label_transfer/multiome_xgboost_preds.csv', sep='\t', index_col=0)
class_df.shape

In [ ]:
class_df.head()

In [ ]:
y_true = class_df['ground_truth']
xgboost_pred = class_df['xgboost_pred']
print(classification_report(y_true, xgboost_pred))

In [ ]:
clusts = sorted(y_true.unique())
mtx = pd.DataFrame(confusion_matrix(y_true, xgboost_pred), index=clusts, columns=clusts)
mtx = mtx.loc[tmp_order, tmp_order]
mtx.head()

In [ ]:
prop_mtx = mtx.div(mtx.sum(axis=0), axis=1)
prop_mtx

In [ ]:
sns.heatmap(mtx, cmap='coolwarm')

In [ ]:
sns.heatmap(prop_mtx, cmap='coolwarm')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
prop_mtx.to_csv(f'{outdir}/snmct_rna_xgboost_conf_mtx.csv', sep='\t')

In [ ]:
y_true = class_df['ground_truth']
classes = np.unique(y_true)
y_true_bin = label_binarize(y_true, classes=classes)  # shape: (n_samples, n_classes)

In [ ]:
y_score = class_df.iloc[:,2:].values
classes = class_df.columns[2:]

In [ ]:
#     precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_score[:, i])
    

In [ ]:
y_true_bin.shape, y_score.shape

In [ ]:
y_true_bin = label_binarize(y_true, classes=classes)

# Plot ROC curves
plt.figure(figsize=(8,6))
for i, class_name in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} (AUC = {roc_auc:.2f})")

plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC (OvR)")
plt.legend()
plt.show()

In [ ]:
!date